In [2]:
pip install datasets transformers

Defaulting to user installation because normal site-packages is not writeable
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ---------------------------------------- 0.0/529.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/529.0 kB ? eta -:--:--
   ------------------- -------------------- 262.1/529.0 kB ? eta -:--:--
   ---------------------------------------- 529.0/529.0 kB 1.3 MB/s  0:00:00
   ---------------------------------------- 0.0/668.2 kB ? eta -:--:--
   --------------- ------------------------ 262.1/668.2 kB ? eta -:--:--
   ------------------------------- -------- 524.3/668.2 kB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 668.2/668.2 kB 1.5 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import re
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoConfig,

    Trainer,
    TrainingArguments,
)

from datasets import (
    Dataset,
    DatasetDict,
    Features, Sequence, ClassLabel, Value
)

C:\Users\lenovo\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
model_name =  "clicknext/phayathaibert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [6]:
train_df = pd.read_csv("train_data.csv")
val_df = pd.read_csv("val_data.csv")

In [19]:
train_df['clean_text'] = train_df['Text'].apply(clean_text)
val_df['clean_text'] = val_df['Text'].apply(clean_text)

In [33]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    
    text = text.strip()
    text = re.sub(r'http[s]?://\S+', ' ลิงก์ ', text)
    text = re.sub(r'\d+', '0', text)
    return text

def preprocess_phayathai(examples: dict) -> dict:
    return tokenizer(
        examples['clean_text'],
        padding='max_length',
        truncation=True,
        max_length= 70,
        return_tensors='pt'
    )   

dataset_dict = DatasetDict({
    'train': Dataset.from_pandas(train_df[['clean_text', 'Label']]),
    'validation': Dataset.from_pandas(val_df[['clean_text', 'Label']])
})

tokenized_datasets = dataset_dict.map(
    preprocess_phayathai, 
    batched=True,
    remove_columns=['clean_text']
)

tokenized_datasets.set_format('torch')
print(tokinized_datasets)

Map: 100%|██████████| 116/116 [00:00<00:00, 5786.00 examples/s]

DatasetDict({
    train: Dataset({
        features: ['Label', 'input_ids', 'attention_mask'],
        num_rows: 541
    })
    validation: Dataset({
        features: ['Label', 'input_ids', 'attention_mask'],
        num_rows: 116
    })
})


In [34]:
# 1. ดึงเอาชุดตัวเลข Token IDs แถวแรกออกมาดู
sample_ids = tokinized_datasets['train'][8]['input_ids']

# 2. สั่งให้ Tokenizer แปลงรหัสตัวเลขกลับมาเป็นข้อความที่มีเครื่องหมาย | แบ่งคำ
print(tokenizer.decode(sample_ids))

<s> คุณได้รับสิทธิ์สินเชื่อ <unk> <<unk>>,<<unk>>คลิก :https://cutt.ly/<unk>f<unk>c<<unk>>n</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>
